# Renaissance — Two-Tower Pretrain Walkthrough

This notebook walks through one complete pretrain iteration using **synthetic data** so it runs on a single CPU or GPU without requiring downloaded datasets.

Topics covered:
1. Building a two-tower model from config
2. Creating a synthetic dataloader
3. Running a pretrain step (MLM + ITM)
4. Saving a checkpoint in safetensors format
5. Loading the checkpoint and running inference
6. Evaluating with the eval harness

> **Requirements:** `pip install -r requirements.txt && pip install -e ..` from the repo root.

## 1. Imports

In [ ]:
import os, sys, tempfile
import torch
import torch.nn.functional as F

# add repo root to path when running from examples/
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), '.'))

from renaissance.modules.renaissance_module import RenaissanceTransformer
from renaissance.modules import renaissance_utils, objectives
from renaissance.eval import evaluate

## 2. Config

We use a minimal two-tower config with tiny encoders and random initialisation so no downloads are needed.

In [ ]:
ALL_LOSSES = {
    "itm": 0, "mlm": 0, "mpp": 0, "vqa": 0, "vcr": 0, "vcr_qar": 0,
    "nlvr2": 0, "irtr": 0, "contras": 0, "snli": 0, "ref": 0, "ref2": 0,
    "mrpc": 0, "rte": 0, "wnli": 0, "sst2": 0, "qqp": 0, "qnli": 0,
    "mnli": 0, "cola": 0, "cifar10": 0,
}

config = {
    # experiment
    "exp_name": "notebook_pretrain",
    "load_path": "",
    "test_only": False,
    "get_recall_metric": False,
    "log_dir": "result",
    "seed": 0,
    "resume_from": None,
    # model
    "model_type": "two-tower",
    "image_encoder": "facebook/deit-tiny-patch16-224",
    "text_encoder": "google/electra-small-discriminator",
    "random_init_vision_encoder": True,   # no download
    "random_init_text_encoder": True,
    "image_encoder_manual_configuration": False,
    "text_encoder_manual_configuration": False,
    "freeze_image_encoder": False,
    "freeze_text_encoder": False,
    "freeze_cross_modal_layers": False,
    "image_encoder_hidden_size": 192,
    "image_encoder_num_heads": 3,
    "image_encoder_num_layers": 12,
    "image_encoder_mlp_ratio": 4,
    "image_encoder_drop_rate": 0.0,
    "image_encoder_embedding_size": 128,
    "text_encoder_hidden_size": 256,
    "text_encoder_num_heads": 4,
    "text_encoder_num_layers": 12,
    "text_encoder_mlp_ratio": 4,
    "text_encoder_drop_rate": 0.0,
    "text_encoder_embedding_size": 64,
    "image_size": 224,
    "original_image_size": 224,
    "patch_size": 16,
    "image_only": False,
    "max_text_len": 16,
    "vocab_size": 30522,
    "cross_layer_hidden_size": 64,
    "num_cross_layers": 2,
    "num_cross_layer_heads": 4,
    "cross_layer_mlp_ratio": 4,
    "cross_layer_drop_rate": 0.0,
    # task
    "loss_names": {**ALL_LOSSES, "mlm": 1, "itm": 1},
    "mlm_prob": 0.15,
    "draw_false_image": 1,
    "draw_false_text": 0,
    "vqav2_label_size": 3129,
    "max_bb": 20,
    "ref_res_head_layers": 2,
    # data / training (used by Trainer)
    "per_gpu_batchsize": 2,
    "batch_size": 2,
    "learning_rate": 1e-4,
    "weight_decay": 0.01,
    "lr_mult_head": 5,
    "lr_mult_cross_modal": 5,
    "end_lr": 0,
    "decay_power": 1,
    "optim_type": "adamw",
    "warmup_steps": 2,
    "max_epoch": 1,
    "max_steps": 5,
    "precision": 32,
    "val_check_interval": 1.0,
    "num_gpus": 1,
    "num_nodes": 1,
    # one-tower (unused here but required by schema)
    "encoder": "google/electra-small-discriminator",
    "random_init_encoder": True,
    "encoder_manual_configuration": False,
    "pooler_type": "double",
    "drop_rate": 0.0,
    "hidden_size": 256,
    "num_heads": 4,
    "num_layers": 6,
    "mlp_ratio": 4,
    "embedding_size": 128,
    "datasets": ["coco"],
}

print("Config ready.")

## 3. Build the model

In [ ]:
model = RenaissanceTransformer(config)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {model.model_type}, {n_params/1e6:.1f}M parameters, device={device}")

## 4. Synthetic dataloader

A real dataloader would read from Arrow files. Here we generate random tensors matching the expected batch format.

In [ ]:
BS = 2
IMG_SIZE = 224
TEXT_LEN = 16
VOCAB = 30522

def make_batch():
    return {
        "image": [torch.randn(BS, 3, IMG_SIZE, IMG_SIZE, device=device)],
        "false_image_0": [torch.randn(BS, 3, IMG_SIZE, IMG_SIZE, device=device)],
        "text": ["a photo of a cat"] * BS,
        "text_ids": torch.randint(1, VOCAB, (BS, TEXT_LEN), device=device),
        "text_labels": torch.full((BS, TEXT_LEN), -100, dtype=torch.long, device=device),
        "text_masks": torch.ones(BS, TEXT_LEN, dtype=torch.long, device=device),
        "text_ids_mlm": torch.randint(1, VOCAB, (BS, TEXT_LEN), device=device),
        "text_labels_mlm": torch.cat([
            torch.randint(1, VOCAB, (BS, 3), device=device),
            torch.full((BS, TEXT_LEN - 3), -100, dtype=torch.long, device=device),
        ], dim=1),
    }

# Wrap in a simple iterable
class SyntheticDataloader:
    def __init__(self, n_batches):
        self.n = n_batches
    def __iter__(self):
        for _ in range(self.n):
            yield make_batch()
    def __len__(self):
        return self.n

train_dataloader = SyntheticDataloader(n_batches=5)
val_dataloader   = SyntheticDataloader(n_batches=2)
print(f"Train: {len(train_dataloader)} batches, Val: {len(val_dataloader)} batches")

## 5. One manual pretrain step

We show a single MLM + ITM step before using the Trainer.

In [ ]:
optimizer, scheduler = renaissance_utils.set_schedule(model, config, max_steps=5)

model.train()
renaissance_utils.set_task(model)

batch = make_batch()
model._log_buffer = {}

out = model(batch)
loss = sum(v for k, v in out.items() if "loss" in k)

loss.backward()
optimizer.step()
scheduler.step()
optimizer.zero_grad()

print(f"MLM loss: {out['mlm_loss'].item():.4f}")
print(f"ITM loss: {out['itm_loss'].item():.4f}")
print(f"Total:    {loss.item():.4f}")

## 6. Save checkpoint

In [ ]:
ckpt_dir = tempfile.mkdtemp(prefix="renaissance_ckpt_")
model.save_pretrained(ckpt_dir)

import os
files = os.listdir(ckpt_dir)
print(f"Saved to {ckpt_dir}: {files}")

## 7. Load checkpoint and run inference

In [ ]:
loaded = RenaissanceTransformer.from_pretrained(ckpt_dir)
loaded = loaded.to(device)
loaded.eval()

batch = make_batch()
with torch.no_grad():
    out = loaded.infer(batch)

print(f"cls_feats shape: {out['cls_feats'].shape}")
print(f"text_feats shape: {out['text_feats'].shape}")
print(f"image_feats shape: {out['image_feats'].shape}")

## 8. Evaluate with the eval harness

The `evaluate` function runs an Evaluator for a specific task over a dataloader and returns a metrics dict.

In [ ]:
metrics = evaluate(loaded, val_dataloader, task="itm")

print("ITM evaluation metrics:")
for k, v in sorted(metrics.items()):
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

## 9. Next steps

- **Real data:** Follow `docs/data-preparation.md` to convert datasets to Arrow format, then point `data.data_root` to the Arrow directory.
- **Full training:** Use `RenaissanceTrainer` (see `renaissance/trainer.py`) or call `python run.py configs/pretrain_two_tower.yaml`.
- **Fine-tuning:** Set `experiment.load_path` to a pretrained checkpoint and change `task.loss_names` to activate the downstream task.
- **Hub:** Push a trained checkpoint with `from renaissance.hub import push_to_hub; push_to_hub(model, 'myuser/mymodel')`.